# Дообучение LoRA (QLoRA) Qwen2.5 на Colab — стили type1/type2

Зеркало локального пайплайна (`scripts/train_lora_3b.py`) для бесплатного GPU Colab (T4, 16 ГБ).

**Порядок работы:**
1. Среда выполнения → Сменить среду выполнения → T4 GPU (бесплатно).
2. Запустите все ячейки (Runtime → Run all).
3. По умолчанию обучается **type1_full (все 160 примеров) на Qwen2.5-7B**.
    Другие варианты — константа `STYLE` в конфиге (`type1` = первые 100, `type2`).
    Для второго стиля поменяйте `STYLE` и перезапустите все ячейки.
4. В конце сессии скачается архив с адаптером — распакуйте его в `models/` локального проекта.

Оценки времени на T4: 7B — ~30–50 мин на стиль (4 эпохи); 14B — ~1,5–2 ч на стиль.

Данные загружаются с вашего компьютера (файлы из data/ проекта). Состав под выбранный STYLE
повторяет обучение финальных 3B-адаптеров либо текущий эксперимент (type1_full).

In [ ]:
# 1. Установка зависимостей: зеркалим рабочее локальное окружение
# (точные версии, проверенные на Windows + CUDA). torch не трогаем —
# на Colab уже стоит CUDA-сборка. Одна команда, чтобы pip-ресолвер собрал
# согласованный набор (unsloth требует unsloth_zoo той же версии).
# Если ниже импорты падают странно — Среда выполнения → Перезапустить и выполнить все.
!pip install -q --upgrade --force-reinstall --no-cache-dir \
  "unsloth==2026.10.5" "unsloth_zoo==2026.10.5" \
  "transformers==5.17.0" "huggingface_hub==1.33.0" "fsspec==2024.9.0" \
  "trl==1.14.2" "peft==0.21.2" "datasets==4.8.5" "accelerate==1.15.0" "bitsandbytes==0.50.2"
!nvidia-smi

## 2. Конфигурация и данные

- `STYLE`: какой набор обучаем — `type1_full` (все 160, стиль @banki_oil, текущий эксперимент),
  `type1` (первые 100, архивный состав финального 3B) или `type2` (@moscowach). Один прогон — один набор.
- Для **14B**: поменяйте `MODEL_ID` на `unsloth/Qwen2.5-14B-Instruct-bnb-4bit` и `MAX_SEQ_LENGTH` на `1024` (иначе OOM на T4).
- `UNSLOTH_CE_LOSS_TARGET_GB` чанкует fused cross-entropy — страховка от OOM на длинных батчах.

In [ ]:
import os
os.environ["UNSLOTH_CE_LOSS_TARGET_GB"] = "0.3"  # должно быть задано ДО импорта unsloth

import json
from google.colab import files

# ===== КОНФИГУРАЦИЯ =====
STYLE = "type1_full"  # "type1_full" (160 примеров), "type1" (первые 100) или "type2"
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
# Для 14B: MODEL_ID = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit" и MAX_SEQ_LENGTH = 1024
MAX_SEQ_LENGTH = 2048  # для 14B ставьте 1024
OUTPUT_DIR = f"lora_{STYLE}_7b_colab"

# Какие файлы загрузить из папки data/ проекта (зависит от STYLE):
# type1_full: все 160 примеров (short+long) — текущий эксперимент для стиля 1.
# type1: первые 100 примеров short — состав финального 3B-адаптера (архивный прогон).
# type2: все 160 примеров (short+long) — состав финального type2-адаптера.
SOURCES = {
    "type1_full": [("train_type1_short.jsonl", None), ("train_type1_long.jsonl", None)],
    "type1": [("train_type1_short.jsonl", 100)],
    "type2": [("train_type2_short.jsonl", None), ("train_type2_long.jsonl", None)],
}[STYLE]

print("⬆️ Загрузите файл(ы) из папки data/ вашего проекта:")
for fname, _ in SOURCES:
    print("   -", fname)
uploaded = files.upload()  # откроется окно выбора файлов с вашего компьютера

items = []
for fname, limit in SOURCES:
    rows = [json.loads(l) for l in uploaded[fname].decode("utf-8").splitlines() if l.strip()]
    if limit:
        rows = rows[:limit]
    print(f"   ➕ {fname}: {len(rows)} примеров")
    for it in rows:
        items.append({"messages": [
            {"role": "system", "content": it["instruction"]},
            {"role": "user", "content": it["input"]},
            {"role": "assistant", "content": it["output"]},
        ]})

print(f"✅ Всего {len(items)} примеров для набора {STYLE}")

## 3. Обучение

Гиперпараметры — те же, что у финальных 3B-адаптеров: r16/α32, dropout 0, LR 2e-4 linear (warmup 5), 4 эпохи, batch 1 × accum 8, fp16, adamw_8bit, seed 42.
Чекпоинты сохраняются после каждой эпохи (хранятся два последних) — если 4-я эпоха окажется переобученной, можно взять 3-ю, как мы делали локально с type2.
Шаблон ChatML берём встроенный из Qwen2.5 — он совпадает с фиксированным шаблоном локального обучения.

In [ ]:
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

dataset = Dataset.from_list(items)

print("🔄 Загрузка модели через Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Автоматически определит float16/bfloat16
    load_in_4bit=True,
    offload_layers="auto",  # страховка для 14B; для 7B планировщик обычно ничего не выгружает
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

def to_text(examples):
    texts = tokenizer.apply_chat_template(
        examples["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": texts}

dataset = dataset.map(to_text, batched=True)
print(f"✅ Датасет готов: {len(dataset)} примеров")

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,  # Эффективный batch = 8
        warmup_steps=5,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=True,  # T4 не поддерживает bf16
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        save_total_limit=2,
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        report_to=[],
    ),
)

stats = trainer.train()
print(f"✅ Обучение завершено. Loss: {stats.training_loss:.4f}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
# 4. Упаковка и скачивание адаптера
import shutil
from google.colab import files

shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
files.download(OUTPUT_DIR + ".zip")
print(f"💾 Распакуйте архив в models/{OUTPUT_DIR} локального проекта")

## 5. Оценка локально (после распаковки в models/)

Имя папки — то же, что `OUTPUT_DIR` (например `lora_type1_full_7b_colab`). Примеры команд (запуск из корня проекта, python — из conda env):

```
PYTHONIOENCODING=utf-8 STYLES=type1 \
  LORA_TYPE1=models/lora_type1_full_7b_colab \
  OUTPUT_TYPE1=outputs_lora_type1_full_7b_t02.txt \
  python scripts/generate_outputs.py
```

Сравнение — с эталоном 3B (`outputs_lora_type1_final_t02.txt`, 16/25 чистых) на тех же 25 тестовых новостях, temp 0.2, seed 42.
Для type2 аналогично через `LORA_TYPE2`/`OUTPUT_TYPE2`.
Если 4-я эпоха подозрительна — распакуйте `checkpoint-*` (эпоха 3) вместо корня адаптера и оцените его.